# Email Spoofing Detection - Exploratory Analysis

Notebook do eksploracyjnej analizy danych e-maili i trenowania modeli.


In [ ]:
# Importy
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from email_parser.parser import EmailParser
from email_parser.header_analyzer import HeaderAnalyzer
from feature_extraction.extractor import FeatureExtractor
from ml_models.train import ModelTrainer
from ml_models.predict import EmailClassifier

%matplotlib inline
sns.set_style('whitegrid')


## 1. Ładowanie i parsowanie przykładowych e-maili


In [ ]:
# Inicjalizacja komponentów
parser = EmailParser()
header_analyzer = HeaderAnalyzer()
feature_extractor = FeatureExtractor()

# Załaduj przykładowe e-maile
with open('../data/sample_emails/sample_legitimate.eml', 'r') as f:
    legitimate_email = f.read()

with open('../data/sample_emails/sample_phishing.eml', 'r') as f:
    phishing_email = f.read()

# Parsuj e-maile
legitimate_data = parser.parse_email_string(legitimate_email)
phishing_data = parser.parse_email_string(phishing_email)

print("=== Legitimate Email ===")
print(f"From: {legitimate_data['from']}")
print(f"Subject: {legitimate_data['subject']}")
print(f"URL count: {len(parser.get_all_urls(legitimate_data))}")

print("\n=== Phishing Email ===")
print(f"From: {phishing_data['from']}")
print(f"Subject: {phishing_data['subject']}")
print(f"URL count: {len(parser.get_all_urls(phishing_data))}")


## 2. Ładowanie i eksploracja zbioru danych

In [ ]:
# Wczytanie zbioru danych
df = pd.read_csv('../data/processed/training_data.csv')

print(f"Rozmiar zbioru danych: {df.shape[0]} próbek, {df.shape[1]} kolumn")
print(f"\nRozkład klas:")
print(df['is_phishing'].value_counts())
print(f"\nProporcje klas:")
print(df['is_phishing'].value_counts(normalize=True).round(4))
print(f"\nStatystyki opisowe:")
df.describe().round(3)

In [ ]:
# Rozkład klas - wizualizacja
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Wykres kołowy
labels = ['Bezpieczne (ham)', 'Phishing/spam']
sizes = df['is_phishing'].value_counts().values
colors = ['#4CAF50', '#F44336']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Rozkład klas w zbiorze danych', fontsize=14, fontweight='bold')

# Wykres słupkowy
df['is_phishing'].value_counts().plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_xticklabels(['Bezpieczne', 'Phishing'], rotation=0)
axes[1].set_title('Liczba próbek w każdej klasie', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Liczba próbek')
for i, v in enumerate(sizes):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../notebooks/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Analiza rozkładów kluczowych cech

In [ ]:
# Porównanie rozkładów wybranych cech między klasami
key_features = [
    'suspicious_keywords_count', 'url_count', 'ip_address_url_count',
    'body_length', 'capital_letter_ratio', 'hop_count',
    'exclamation_count', 'anomaly_score', 'avg_hop_delay'
]

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()

for idx, feature in enumerate(key_features):
    if feature in df.columns:
        ham = df[df['is_phishing'] == 0][feature]
        spam = df[df['is_phishing'] == 1][feature]
        
        axes[idx].hist(ham, bins=30, alpha=0.6, label='Bezpieczne', color='#4CAF50', density=True)
        axes[idx].hist(spam, bins=30, alpha=0.6, label='Phishing', color='#F44336', density=True)
        axes[idx].set_title(feature, fontsize=11, fontweight='bold')
        axes[idx].legend(fontsize=9)
        axes[idx].set_ylabel('Gęstość')

plt.suptitle('Rozkłady kluczowych cech: Bezpieczne vs Phishing', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../notebooks/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Box-ploty kluczowych cech
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

box_features = [
    'suspicious_keywords_count', 'url_count', 'body_length',
    'capital_letter_ratio', 'exclamation_count', 'hop_count'
]

for idx, feature in enumerate(box_features):
    if feature in df.columns:
        data_to_plot = [
            df[df['is_phishing'] == 0][feature].values,
            df[df['is_phishing'] == 1][feature].values
        ]
        bp = axes[idx].boxplot(data_to_plot, labels=['Bezpieczne', 'Phishing'],
                               patch_artist=True)
        bp['boxes'][0].set_facecolor('#C8E6C9')
        bp['boxes'][1].set_facecolor('#FFCDD2')
        axes[idx].set_title(feature, fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('Wartość')

plt.suptitle('Porównanie cech między klasami (box-plot)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../notebooks/feature_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Macierz korelacji cech

In [ ]:
# Heatmapa korelacji
plt.figure(figsize=(20, 16))
corr_matrix = df.corr()

# Maska trójkąta górnego
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5, square=True)
plt.title('Macierz korelacji cech', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Top korelacje z etykietą is_phishing
target_corr = corr_matrix['is_phishing'].drop('is_phishing').abs().sort_values(ascending=False)
print("Top 15 cech najbardziej skorelowanych z is_phishing:")
for feat, corr_val in target_corr.head(15).items():
    direction = corr_matrix.loc[feat, 'is_phishing']
    sign = '+' if direction > 0 else '-'
    print(f"  {feat:40s} {sign}{corr_val:.4f}")

## 5. Analiza cech autentykacji (SPF, DKIM, DMARC)

In [ ]:
# Analiza cech autentykacji
auth_features = ['spf_pass', 'spf_fail', 'spf_softfail', 'dkim_valid', 'dkim_invalid',
                 'dmarc_exists', 'dmarc_policy_none', 'dmarc_policy_quarantine', 'dmarc_policy_reject']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for idx, feature in enumerate(auth_features):
    if feature in df.columns:
        ct = pd.crosstab(df[feature], df['is_phishing'], normalize='columns')
        ct.columns = ['Bezpieczne', 'Phishing']
        ct.plot(kind='bar', ax=axes[idx], color=['#4CAF50', '#F44336'])
        axes[idx].set_title(feature, fontsize=11, fontweight='bold')
        axes[idx].set_ylabel('Proporcja')
        axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=0)
        axes[idx].legend(fontsize=8)

plt.suptitle('Cechy autentykacji: SPF / DKIM / DMARC', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../notebooks/auth_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Trening i ewaluacja modeli

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_curve, auc, precision_recall_curve, 
                             f1_score, accuracy_score, precision_score, recall_score)
from imblearn.over_sampling import SMOTE

# Przygotowanie danych
y = df['is_phishing']
X = df.drop(columns=['is_phishing'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Zbiór treningowy: {X_train.shape[0]} próbek")
print(f"Zbiór testowy:    {X_test.shape[0]} próbek")
print(f"Rozkład train: {y_train.value_counts().to_dict()}")
print(f"Rozkład test:  {y_test.value_counts().to_dict()}")

# SMOTE - balansowanie klas
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print(f"\nPo SMOTE: {pd.Series(y_train_bal).value_counts().to_dict()}")

# Skalowanie
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Trening Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=20, min_samples_split=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train_bal)

# Predykcje
y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

print("=" * 60)
print("RANDOM FOREST - Raport klasyfikacji")
print("=" * 60)
print(classification_report(y_test, y_pred_rf, 
                            target_names=['Bezpieczne', 'Phishing']))

print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_rf):.4f}")

In [ ]:
# Trening Isolation Forest
if_model = IsolationForest(
    n_estimators=100, contamination=0.1, max_samples=256,
    random_state=42, n_jobs=-1
)
if_model.fit(X_train_scaled)

# Predykcje — mapowanie: -1 (anomalia) → 1 (phishing), 1 (normal) → 0
y_pred_if_raw = if_model.predict(X_test_scaled)
y_pred_if = np.where(y_pred_if_raw == -1, 1, 0)
if_scores = if_model.score_samples(X_test_scaled)

print("=" * 60)
print("ISOLATION FOREST - Raport klasyfikacji")
print("=" * 60)
print(classification_report(y_test, y_pred_if,
                            target_names=['Bezpieczne', 'Phishing']))

print(f"Accuracy:  {accuracy_score(y_test, y_pred_if):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_if):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_if):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_if):.4f}")

## 7. Wizualizacja wyników — macierze pomyłek

In [ ]:
# Macierze pomyłek — porównanie obu modeli
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Bezpieczne', 'Phishing'],
            yticklabels=['Bezpieczne', 'Phishing'])
axes[0].set_title('Random Forest\nMacierz pomyłek', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Rzeczywista klasa')
axes[0].set_xlabel('Predykcja')

# Isolation Forest
cm_if = confusion_matrix(y_test, y_pred_if)
sns.heatmap(cm_if, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Bezpieczne', 'Phishing'],
            yticklabels=['Bezpieczne', 'Phishing'])
axes[1].set_title('Isolation Forest\nMacierz pomyłek', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Rzeczywista klasa')
axes[1].set_xlabel('Predykcja')

plt.tight_layout()
plt.savefig('../notebooks/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Krzywa ROC i krzywa Precision-Recall

In [ ]:
# Krzywa ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC — Random Forest
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)
roc_auc_rf = auc(fpr_rf, tpr_rf)

# ROC — Isolation Forest (używamy anomaly scores odwróconych)
fpr_if, tpr_if, _ = roc_curve(y_test, -if_scores)
roc_auc_if = auc(fpr_if, tpr_if)

axes[0].plot(fpr_rf, tpr_rf, color='#1565C0', lw=2, label=f'Random Forest (AUC = {roc_auc_rf:.3f})')
axes[0].plot(fpr_if, tpr_if, color='#FF6F00', lw=2, label=f'Isolation Forest (AUC = {roc_auc_if:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Losowy klasyfikator')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Krzywa ROC', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall — Random Forest
precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_pred_proba_rf)
pr_auc_rf = auc(recall_rf, precision_rf)

# Precision-Recall — Isolation Forest
precision_if, recall_if, _ = precision_recall_curve(y_test, -if_scores)
pr_auc_if = auc(recall_if, precision_if)

axes[1].plot(recall_rf, precision_rf, color='#1565C0', lw=2, label=f'Random Forest (AUC = {pr_auc_rf:.3f})')
axes[1].plot(recall_if, precision_if, color='#FF6F00', lw=2, label=f'Isolation Forest (AUC = {pr_auc_if:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Krzywa Precision-Recall', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../notebooks/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Random Forest  — ROC AUC: {roc_auc_rf:.4f}, PR AUC: {pr_auc_rf:.4f}")
print(f"Isolation Forest — ROC AUC: {roc_auc_if:.4f}, PR AUC: {pr_auc_if:.4f}")

## 9. Feature Importance — najważniejsze cechy

In [ ]:
# Feature importance z Random Forest
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(12, 14))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(importance_df)))

plt.barh(importance_df['feature'], importance_df['importance'], color=colors)
plt.xlabel('Ważność cechy (importance)', fontsize=12)
plt.title('Random Forest — ważność cech', fontsize=16, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../notebooks/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 20 najważniejszych cech:")
for _, row in importance_df.tail(20).iloc[::-1].iterrows():
    print(f"  {row['feature']:40s} {row['importance']:.4f}")

## 10. Porównanie modeli — podsumowanie

In [ ]:
# Porównanie metryk obu modeli
comparison = pd.DataFrame({
    'Metryka': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC AUC'],
    'Random Forest': [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf),
        roc_auc_rf
    ],
    'Isolation Forest': [
        accuracy_score(y_test, y_pred_if),
        precision_score(y_test, y_pred_if),
        recall_score(y_test, y_pred_if),
        f1_score(y_test, y_pred_if),
        roc_auc_if
    ]
}).set_index('Metryka')

print("=" * 60)
print("PORÓWNANIE MODELI")
print("=" * 60)
print(comparison.round(4).to_string())

# Wykres porównawczy
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparison.index))
width = 0.35

bars1 = ax.bar(x - width/2, comparison['Random Forest'], width, label='Random Forest', color='#1565C0')
bars2 = ax.bar(x + width/2, comparison['Isolation Forest'], width, label='Isolation Forest', color='#FF6F00')

ax.set_ylabel('Wartość metryki')
ax.set_title('Porównanie skuteczności modeli', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison.index, rotation=15)
ax.legend()
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../notebooks/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Wnioski

**Random Forest** osiąga znacznie lepsze wyniki klasyfikacji niż Isolation Forest, co jest oczekiwane — RF jest modelem nadzorowanym, trenowanym na etykietowanych danych.

**Isolation Forest** jako metoda detekcji anomalii ma wysoką precyzję (mało fałszywych alarmów), ale niski recall — wykrywa tylko najbardziej oczywiste anomalie. Jest przydatny jako dodatkowa warstwa weryfikacji.

Kluczowe cechy dla wykrywania phishingu:
- Cechy treściowe (podejrzane słowa kluczowe, stosunek wielkich liter)
- Cechy URL-i (adresy IP w URL, skracacze linków)
- Cechy autentykacji (wynik SPF, ważność DKIM, polityka DMARC)
- Cechy nagłówków (niezgodności From/Return-Path, anomalie czasowe)

System łączy oba modele — decyzja oparta na głosowaniu większościowym z wagami pewności.